# Week 10: GPT-2 (124M), first training steps

Run this notebook once on **T4**, download the results, then once on **A100**.
Choose **Runtime → Change runtime type → Hardware accelerator** before starting.
This runs the full 124,439,808-parameter model, including a 300-update fixed-batch
diagnostic. It does not train GPT-2 on an entire corpus. GPU execution uses your
Colab allocation; no subscription change or new purchase is needed.

This notebook embeds the project source snapshot with SHA-256 checks. It works
without pushing code to GitHub. No private Week 8 corpus or Google Drive mount is
needed. The exported ZIP contains metrics, samples, plots and source hashes.


## 1. Unpack the checked source snapshot and install dependencies


In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, zipfile
from pathlib import Path

ROOT = Path('/content/neuron-to-gpt-week10')
ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE_SHA256 = '0b5cc3a9a3d5cdc1c727db1cc6c7e5f5bc9a1d31add90a0e3beca754e49ca75c'
SOURCE_HASHES = {'model.py': 'c3880320d61666d7c79f861dba74fb242ad09085fb7e03dd45ea64dde55f9535', 'run_experiments.py': '57ca7e634a14c665712ead551d142646dd82fc19dc9667d09a5eaeea541c01ea', 'test_gpt2.py': 'd81b5db67ca1e8f2920458f9cb1c5d50170b9896987d5c3c85c5c91bec60cdc0', 'requirements.txt': '68930bfe39dec13ca0cd507379425475407618bf4af86d9e2fbe496d838bd38a'}
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == BUNDLE_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    expected = {'week10/' + name for name in SOURCE_HASHES}
    assert set(archive.namelist()) == expected
    for name, checksum in SOURCE_HASHES.items():
        raw = archive.read('week10/' + name)
        assert hashlib.sha256(raw).hexdigest() == checksum
        target = ROOT / 'week10' / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(raw)
os.environ['HF_HOME'] = str(ROOT / 'week10/.cache/huggingface')
os.environ['TIKTOKEN_CACHE_DIR'] = str(ROOT / 'week10/.cache/tiktoken')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'week10/requirements.txt')], check=True)
print('Verified source bundle:', BUNDLE_SHA256)
print(json.dumps(SOURCE_HASHES, indent=2))


## 2. Verify the selected GPU


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then reconnect.'
GPU_NAME = torch.cuda.get_device_name(0)
if 'A100' in GPU_NAME:
    RUN_NAME, PRECISION = 'a100', 'bf16'
elif 'T4' in GPU_NAME:
    RUN_NAME, PRECISION = 't4', 'fp16'
else:
    raise RuntimeError(f'Select a T4 or A100 for this comparison; connected: {GPU_NAME}')
RESULTS = ROOT / 'week10/results' / RUN_NAME
RESULTS.mkdir(parents=True, exist_ok=True)
print('GPU:', GPU_NAME, '| torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('Training:', PRECISION, '| logits parity and loss evaluation: FP32')
print('Results:', RESULTS)
subprocess.run(['nvidia-smi'], check=True)


## 3. Run the independent model checks
These tests use tiny models without downloading pretrained weights.


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'week10', '-p', 'test_*.py'], cwd=ROOT, check=True)


## 4. Inspect published weights, compare logits, sample, and score translations
The roughly 550 MB public safetensors weights download on first run. FP32/eager parity is checked before samples.


In [ ]:
# Full 124M model. The public weights are pinned to one published revision.
subprocess.run([
    sys.executable, '-u', 'week10/run_experiments.py', '--task', 'pretrained',
    '--device', 'cuda', '--output-dir', str(RESULTS), '--new-tokens', '64',
    '--cache-dir', str(ROOT / 'week10/.cache/huggingface'),
    '--revision', '607a30d783dfa663caf39e06633721c8d4cfcd7e',
], cwd=ROOT, check=True)


## 5. Random initialization and 300 updates on exactly one batch
T4 uses FP16 + GradScaler; A100 uses BF16. Every recorded evaluation uses FP32. Progress is saved every 25 steps. The command fails if the final loss does not meet the diagnostic threshold.


In [ ]:
# Fresh random weights; this is not fine-tuning the pretrained model.
# Same fixed batch and schedule on both GPUs. Precision differs, so timings
# are not a controlled hardware-only benchmark. No compile/FlashAttention/DDP.
subprocess.run([
    sys.executable, '-u', 'week10/run_experiments.py', '--task', 'overfit',
    '--device', 'cuda', '--output-dir', str(RESULTS), '--precision', PRECISION,
    '--steps', '300', '--batch-size', '4', '--block-size', '32',
    '--learning-rate', '0.001', '--eval-interval', '25', '--seed', '1337',
], cwd=ROOT, check=True)


## 6. Inspect the measured results


In [ ]:
from IPython.display import display, Markdown, Image
pretrained = json.loads((RESULTS / 'pretrained_metrics.json').read_text())
overfit = json.loads((RESULTS / 'overfit_metrics.json').read_text())
assert pretrained['logit_parity']['all_passed']
assert overfit['evidence'] == 'full_gpt2_124m'
assert overfit['status'] == 'completed' and overfit['overfit_passed']
display(Markdown((RESULTS / 'submission_report.md').read_text()))
display(Image(filename=str(RESULTS / 'position_embeddings.png')))
display(Image(filename=str(RESULTS / 'overfit_loss.png')))
print('Initial loss:', overfit['initial_loss'], '| final:', overfit['final_loss'])
print('Completed optimizer updates:', overfit['optimizer_updates'])


## 7. Download results before switching GPU
After saving the ZIP, change the GPU, reconnect, and rerun cells 1–7 for the other device. A runtime change can erase `/content` files.


In [ ]:
from google.colab import files
export = ROOT / f'week10_{RUN_NAME}_results.zip'
manifest = {'gpu': GPU_NAME, 'source_bundle_sha256': BUNDLE_SHA256, 'source_hashes': SOURCE_HASHES}
(RESULTS / 'source_manifest.json').write_text(json.dumps(manifest, indent=2) + chr(10))
with zipfile.ZipFile(export, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.iterdir()):
        if path.is_file() and path.suffix in {'.json', '.tsv', '.txt', '.md', '.png'}:
            archive.write(path, arcname=RUN_NAME + '/' + path.name)
print('Export:', export, '| bytes:', export.stat().st_size)
print('Save this ZIP before changing GPU/runtime. Model weights and caches are excluded.')
files.download(str(export))
